In [16]:

# %%
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# %% [markdown]
# ## 2. 配置参数（根据需要修改）

# %%
# ========== 配置参数 ==========
INPUT_PATH = "/mnt/data/zwl/data/mixed.parquet"  # 输入文件路径
OUTPUT_DIR = "/mnt/data/zwl/data/rl"                     # 输出目录
PREFIX = "qwen3_4b_grpo_mixed"                                 # 输出文件前缀
VAL_RATIO = 0.1                                         # 验证集比例
MIN_VAL_SAMPLES = 64                                      # 最小验证集样本数
SEED = 42                                                 # 随机种子
# =============================

# %% [markdown]
# ## 3. 辅助函数

# %%
def to_python_obj(value: Any) -> Any:
    """将各种格式的值转换为 Python 对象"""
    if value is None:
        return None
    if isinstance(value, (list, dict)):
        return value
    if isinstance(value, str):
        text = value.strip()
        if (text.startswith("[") and text.endswith("]")) or (text.startswith("{") and text.endswith("}")):
            try:
                return json.loads(text)
            except Exception:
                return value
        return value
    if hasattr(value, "tolist") and not isinstance(value, (bytes, bytearray)):
        try:
            return value.tolist()
        except Exception:
            pass
    return value


def normalize_role(role: Any) -> str:
    """标准化角色名称"""
    role = str(role).strip().lower()
    mapping = {
        "human": "user",
        "user": "user",
        "assistant": "assistant",
        "gpt": "assistant",
        "bot": "assistant",
        "system": "system",
    }
    return mapping.get(role, role)


def normalize_message(message: Any) -> dict[str, str]:
    """标准化消息格式"""
    if not isinstance(message, dict):
        raise TypeError(f"Unsupported message format: {type(message)}")

    role = message.get("role", message.get("from", message.get("speaker", "user")))
    content = message.get("content", message.get("value", message.get("text", "")))

    if isinstance(content, list):
        content = "\n".join(str(x) for x in content)

    return {
        "role": normalize_role(role),
        "content": "" if content is None else str(content),
    }


def build_rl_record(row: pd.Series, row_idx: int, src_path: str) -> dict[str, Any] | None:
    """构建 RL 训练记录"""
    messages = row.get("messages", row.get("prompt", None))
    messages = to_python_obj(messages)
    if not isinstance(messages, list) or len(messages) < 2:
        return None

    normalized_messages: list[dict[str, str]] = []
    for msg in messages:
        msg = to_python_obj(msg)
        if msg is None:
            continue
        normalized_messages.append(normalize_message(msg))

    if len(normalized_messages) < 2:
        return None

    # 找到最后一条非空的 assistant 消息
    answer_idx = None
    for idx in range(len(normalized_messages) - 1, -1, -1):
        msg = normalized_messages[idx]
        if msg["role"] == "assistant" and msg["content"].strip():
            answer_idx = idx
            break

    if answer_idx is None or answer_idx == 0:
        return None

    prompt = [m for m in normalized_messages[:answer_idx] if m["content"].strip()]
    ground_truth = normalized_messages[answer_idx]["content"].strip()
    if not prompt or not ground_truth:
        return None

    return {
        "data_source": "custom_sft_exact_match",
        "prompt": prompt,
        "reward_model": {"style": "rule", "ground_truth": ground_truth},
        "extra_info": {
            "source_file": src_path,
            "row_idx": int(row_idx),
        },
    }

# %% [markdown]
# ## 4. 执行转换

# %%
# 检查输入文件
input_path = Path(INPUT_PATH)
if not input_path.exists():
    raise FileNotFoundError(f"Input parquet not found: {input_path}")

print(f"✅ 找到输入文件: {input_path}")

# 创建输出目录
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"✅ 输出目录已准备: {output_dir}")

# 读取数据
df = pd.read_parquet(input_path)
print(f"✅ 读取了 {len(df)} 行数据")
print(f"   列名: {list(df.columns)}")

# 转换数据
records: list[dict[str, Any]] = []
skipped = 0

for row_idx, row in df.iterrows():
    record = build_rl_record(row, row_idx, str(input_path))
    if record is None:
        skipped += 1
        continue
    records.append(record)

print(f"✅ 转换完成")
print(f"   可用样本: {len(records)}")
print(f"   跳过样本: {skipped}")

# 检查是否有可用样本
if not records:
    raise ValueError(
        "No usable samples were produced. "
        "Expected each row to contain a `messages` column and end with a non-empty assistant answer."
    )

# 创建 DataFrame
out_df = pd.DataFrame(records)
print(f"✅ 创建 DataFrame: {len(out_df)} 行")

# %% [markdown]
# ## 5. 划分训练集和验证集

# %%
train_path = output_dir / f"{PREFIX}_train.parquet"
val_path = output_dir / f"{PREFIX}_val.parquet"

if len(out_df) == 1:
    # 只有一个样本，训练集和验证集共用
    train_df = out_df.copy()
    val_df = out_df.copy()
    print("⚠️ 只有一个样本，训练集和验证集共用")
else:
    # 随机划分
    rng = np.random.default_rng(SEED)
    perm = rng.permutation(len(out_df))
    
    # 计算验证集大小
    suggested_val = max(1, int(len(out_df) * VAL_RATIO))
    warm_val = min(MIN_VAL_SAMPLES, max(1, len(out_df) // 10))
    val_size = min(max(suggested_val, warm_val), len(out_df) - 1)
    
    val_idx = perm[:val_size]
    train_idx = perm[val_size:]
    
    train_df = out_df.iloc[train_idx].reset_index(drop=True)
    val_df = out_df.iloc[val_idx].reset_index(drop=True)
    
    print(f"验证集大小: {len(val_df)} (占总数的 {len(val_df)/len(out_df)*100:.1f}%)")

# 保存文件
train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path, index=False)

print(f"\n✅ 文件已保存:")
print(f"   训练集: {train_path} ({len(train_df)} 条)")
print(f"   验证集: {val_path} ({len(val_df)} 条)")

# %% [markdown]
# ## 6. 验证输出

# %%
# 查看训练集样例
print("\n📊 训练集样例:")
print("=" * 50)
sample = train_df.iloc[0]
print(f"data_source: {sample['data_source']}")
print(f"prompt: {sample['prompt'][:2]}...")  # 只显示前2条消息
print(f"reward_model: {sample['reward_model']}")
print(f"extra_info: {sample['extra_info']}")

# %% [markdown]
# ## 7. 数据统计

# %%
print("\n📈 数据统计:")
print("=" * 50)
print(f"原始数据行数      : {len(df)}")
print(f"可用数据行数      : {len(out_df)}")
print(f"跳过行数          : {skipped}")
print(f"训练集样本数      : {len(train_df)}")
print(f"验证集样本数      : {len(val_df)}")

# 显示 prompt 长度分布
prompt_lengths = [len(p) for p in train_df['prompt'].values]
print(f"\nPrompt 消息数量统计:")
print(f"  最小: {min(prompt_lengths)}")
print(f"  最大: {max(prompt_lengths)}")
print(f"  平均: {sum(prompt_lengths)/len(prompt_lengths):.1f}")

# 显示 ground_truth 长度分布
gt_lengths = [len(sample['reward_model']['ground_truth']) for _, sample in train_df.iterrows()]
print(f"\nGround Truth 长度统计:")
print(f"  最小: {min(gt_lengths)}")
print(f"  最大: {max(gt_lengths)}")
print(f"  平均: {sum(gt_lengths)/len(gt_lengths):.1f}")

# %% [markdown]
# ## 完成！

✅ 找到输入文件: /mnt/data/zwl/data/mixed.parquet
✅ 输出目录已准备: /mnt/data/zwl/data/rl
✅ 读取了 1000 行数据
   列名: ['messages']
✅ 转换完成
   可用样本: 1000
   跳过样本: 0
✅ 创建 DataFrame: 1000 行
验证集大小: 100 (占总数的 10.0%)

✅ 文件已保存:
   训练集: /mnt/data/zwl/data/rl/qwen3_4b_grpo_mixed_train.parquet (900 条)
   验证集: /mnt/data/zwl/data/rl/qwen3_4b_grpo_mixed_val.parquet (100 条)

📊 训练集样例:
data_source: custom_sft_exact_match
prompt: [{'role': 'user', 'content': '请解答以下问题，并给出详细推导或分析过程：\nIn the carotenoid enrichment study, once the carotenoid content $C$ in the UAE-enriched flaxseed oil has been established via the spectrophotometric formula derived previously,\n$$C = \\frac{V(A_s - A_b)}{100\\,\\varepsilon\\, W},$$\nthe next task is to design the experimental matrix used to optimize $C$ with respect to the three process variables—extraction time $t$, feed-to-oil ratio $R$, and ultrasonic amplitude $A$—using a Box-Behnken response surface design.\n\nA Box-Behnken design (BBD) for $K$ factors is constructed by placing experimenta

In [17]:
import pyarrow.parquet as pq

# 读取 Parquet 文件
table = pq.read_table('/mnt/data/zwl/data/rl/qwen3_4b_grpo_edge_only_train.parquet')

# 转换为 pandas DataFrame
df = table.to_pandas()

print(f"行数: {len(df)}")
print(f"列名: {df.columns.tolist()}")

行数: 900
列名: ['data_source', 'prompt', 'reward_model', 'extra_info']


In [19]:
import pyarrow.parquet as pq
import pandas as pd
import json

# 读取 Parquet 文件
table = pq.read_table('/mnt/data/zwl/data/rl/qwen3_4b_grpo_edge_only_train.parquet')
df = table.to_pandas()

print(f"行数: {len(df)}")
print(f"列名: {df.columns.tolist()}")
print(f"数据形状: {df.shape}")
print("\n" + "="*80)
print("前 1 条数据（完整内容）:")
print("="*80)

# 设置 pandas 显示选项，显示完整内容
pd.set_option('display.max_colwidth', None)  # 不限制列宽
pd.set_option('display.max_rows', None)       # 不限制行数
pd.set_option('display.width', None)          # 不限制宽度
pd.set_option('display.max_seq_items', None)  # 不限制序列项

# 方法1: 直接显示
print(df.head(1))

print("\n" + "="*80)
print("逐列详细查看:")
print("="*80)

# 方法2: 逐列显示完整内容
for col in df.columns:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    print(f"前1条内容:")
    try:
        # 尝试美化显示
        value = df[col].iloc[0]
        if isinstance(value, (dict, list)):
            print(json.dumps(value, ensure_ascii=False, indent=2))
        else:
            print(value)
    except Exception as e:
        print(f"无法显示: {e}")

print("\n" + "="*80)
print("数据统计信息:")
print("="*80)
print(df.describe())

# 如果有字符串列，显示长度信息
print("\n" + "="*80)
print("字符串列长度信息:")
print("="*80)
for col in df.select_dtypes(include=['object']).columns:
    try:
        print(f"{col}: 最小长度={df[col].str.len().min()}, 最大长度={df[col].str.len().max()}, 平均长度={df[col].str.len().mean():.2f}")
    except:
        pass

行数: 900
列名: ['data_source', 'prompt', 'reward_model', 'extra_info']
数据形状: (900, 4)

前 1 条数据（完整内容）:
              data_source  \
0  custom_sft_exact_match   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          